##### 1. Setup Environment & Imports
Cài đặt môi trường và nhập các thư viện cần thiết

In [1]:
# Import thư viện cần thiết
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os
import warnings

warnings.filterwarnings('ignore')

# Cấu hình Pandas display
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

print("Thư viện đã được nhập thành công")

Thư viện đã được nhập thành công


##### 2. Tìm kiếm và đọc file CSV

In [2]:
# Lấy đường dẫn thư mục hiện tại
current_dir = Path.cwd()
base_dir = Path(r"d:\My Stuff\KHDL28B\Deep Learning")
work_dir = base_dir / 'data'

print(f"Thư mục dữ liệu: {work_dir}")
print(f"Đang sử dụng file: raw_data.csv\n")

csv_path = work_dir / 'raw_data.csv'

# Đọc file CSV - Bỏ qua metadata
print("Đang đọc file CSV...\n")

try:
    # File này có metadata ở 2 dòng đầu, nên bỏ qua dòng 0-2
    df = pd.read_csv(csv_path, skiprows=3)
    
    print("Đã đọc file CSV thành công!")
    print("\nThông tin cơ bản về dữ liệu:\n")
    print(f"Kích thước: {df.shape[0]} hàng × {df.shape[1]} cột")
    
except Exception as e:
    print(f"Lỗi: {e}")

Thư mục dữ liệu: d:\My Stuff\KHDL28B\Deep Learning\data
Đang sử dụng file: raw_data.csv

Đang đọc file CSV...

Đã đọc file CSV thành công!

Thông tin cơ bản về dữ liệu:

Kích thước: 92640 hàng × 11 cột


##### 3. Xem dữ liệu ban đầu

In [3]:
df

,time,pm10 (μg/m³),pm2_5 (μg/m³),carbon_monoxide (μg/m³),nitrogen_dioxide (μg/m³),sulphur_dioxide (μg/m³),ozone (μg/m³),aerosol_optical_depth (),dust (μg/m³),uv_index (),uv_index_clear_sky ()
0,2016-01-01T00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-01-01T01:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2016-01-01T02:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2016-01-01T03:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2016-01-01T04:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
92635,2026-07-26T19:00,13.3,11.2,291.0,4.1,2.6,87.0,0.24,2.0,0.0,0.0
92636,2026-07-26T20:00,15.2,12.9,320.0,5.0,3.0,76.0,0.22,2.0,0.0,0.0
92637,2026-07-26T21:00,15.7,13.8,343.0,6.1,3.6,61.0,0.20,3.0,0.0,0.0
92638,2026-07-26T22:00,15.0,13.1,345.0,6.8,3.9,49.0,0.19,3.0,0.0,0.0


##### 4. Kiểm tra thông tin chi tiết về cột dữ liệu

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92640 entries, 0 to 92639
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   time                      92640 non-null  object 
 1   pm10 (μg/m³)              34865 non-null  float64
 2   pm2_5 (μg/m³)             34865 non-null  float64
 3   carbon_monoxide (μg/m³)   34865 non-null  float64
 4   nitrogen_dioxide (μg/m³)  34865 non-null  float64
 5   sulphur_dioxide (μg/m³)   34865 non-null  float64
 6   ozone (μg/m³)             34865 non-null  float64
 7   aerosol_optical_depth ()  34865 non-null  float64
 8   dust (μg/m³)              34865 non-null  float64
 9   uv_index ()               34865 non-null  float64
 10  uv_index_clear_sky ()     34865 non-null  float64
dtypes: float64(10), object(1)
memory usage: 7.8+ MB


##### 5. Loại bỏ các hàng có toàn bộ giá trị NaN (trừ cột time)

In [5]:
cols_to_check = [col for col in df.columns if col != 'time']
df_copy = df.dropna(subset=cols_to_check, how='all')

##### 6. Loại bỏ các cột có chứa giá trị NaN

In [6]:
df_copy = df_copy.dropna(axis=1)

##### 7. Chuyển đổi cột time thành datetime và tách thành các cột ngày, tháng, năm, giờ

In [7]:
df_copy['time'] = pd.to_datetime(df_copy['time'])
df_copy['day'] = df_copy['time'].dt.day
df_copy['month'] = df_copy['time'].dt.month
df_copy['year'] = df_copy['time'].dt.year
df_copy['hour'] = df_copy['time'].dt.hour

cols = df_copy.columns.tolist()
time_idx = cols.index('time')
cols = cols[:time_idx+1] + ['day', 'month', 'year', 'hour'] + [c for c in cols[time_idx+1:] if c not in ['day', 'month', 'year', 'hour']]
df_copy = df_copy[cols]
df_copy.reset_index(drop=True, inplace=True)

##### 8. Xem dữ liệu sau khi làm sạch và chuyển đổi

In [8]:
df_copy

,time,day,month,year,hour,pm10 (μg/m³),pm2_5 (μg/m³),carbon_monoxide (μg/m³),nitrogen_dioxide (μg/m³),sulphur_dioxide (μg/m³),ozone (μg/m³),aerosol_optical_depth (),dust (μg/m³),uv_index (),uv_index_clear_sky ()
0,2022-08-04 07:00:00,4,8,2022,7,11.8,8.3,230.0,3.1,1.0,32.0,0.08,0.0,1.10,1.20
1,2022-08-04 08:00:00,4,8,2022,8,10.2,7.1,216.0,2.6,0.9,37.0,0.08,0.0,3.25,3.55
2,2022-08-04 09:00:00,4,8,2022,9,9.3,6.5,195.0,1.8,0.8,43.0,0.08,0.0,6.10,6.95
3,2022-08-04 10:00:00,4,8,2022,10,9.7,6.8,174.0,1.1,0.8,51.0,0.08,0.0,7.85,10.45
4,2022-08-04 11:00:00,4,8,2022,11,11.0,7.7,168.0,0.9,0.8,55.0,0.08,0.0,9.55,12.80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34860,2026-07-26 19:00:00,26,7,2026,19,13.3,11.2,291.0,4.1,2.6,87.0,0.24,2.0,0.00,0.00
34861,2026-07-26 20:00:00,26,7,2026,20,15.2,12.9,320.0,5.0,3.0,76.0,0.22,2.0,0.00,0.00
34862,2026-07-26 21:00:00,26,7,2026,21,15.7,13.8,343.0,6.1,3.6,61.0,0.20,3.0,0.00,0.00
34863,2026-07-26 22:00:00,26,7,2026,22,15.0,13.1,345.0,6.8,3.9,49.0,0.19,3.0,0.00,0.00


##### 9. Kiểm tra xem dữ liệu time có liên tục không (cách 1 giờ)

In [9]:
def check_continuous_time(df):
    if 'time' not in df.columns:
        return False
    
    time_series = pd.to_datetime(df['time'])
    time_diff = time_series.diff()
    expected_diff = pd.Timedelta(hours=1)
    
    is_continuous = (time_diff.iloc[1:] == expected_diff).all()
    return is_continuous

check_continuous_time(df_copy)

True

##### 10. Lưu dữ liệu đã làm sạch thành file CSV mới

In [10]:
output_path = work_dir / 'cleaned_data.csv'
df_copy.to_csv(output_path, index=False, encoding='utf-8')
print(f"Dữ liệu đã được lưu tại: {output_path}")

Dữ liệu đã được lưu tại: d:\My Stuff\KHDL28B\Deep Learning\data\cleaned_data.csv
